# Peer University Benchmarking Analysis

**Objective:** Compare AUB citation patterns against peer institutions (Lehigh, Marquette, Villanova)

## Data Sources:
- AUB: `data/processed/cleaned_data.csv` (already cleaned)
- Lehigh: `data/raw/lehigh.csv`
- Marquette: `data/raw/marquette.csv`
- Villanova: `data/raw/villanova.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

%matplotlib inline

## 1. Load All Data Sources

In [ ]:
# Load AUB data (already cleaned)
aub = pd.read_csv('../data/processed/cleaned_data.csv')
print(f"AUB: {len(aub)} publications")
print(f"Columns: {list(aub.columns)}")
aub.head()

In [ ]:
# DIAGNOSTIC: Load peer universities and inspect structure
# These CSVs appear to have unusual formatting - let's investigate

print("=" * 60)
print("LOADING PEER UNIVERSITY DATA")
print("=" * 60)

# Try loading with default settings first
lehigh_raw = pd.read_csv('../data/raw/lehigh.csv')
marquette_raw = pd.read_csv('../data/raw/marquette.csv')
villanova_raw = pd.read_csv('../data/raw/villanova.csv')

print(f"\nLehigh shape: {lehigh_raw.shape}")
print(f"Marquette shape: {marquette_raw.shape}")
print(f"Villanova shape: {villanova_raw.shape}")

print("\n" + "=" * 60)
print("FIRST 5 ROWS OF LEHIGH (to understand structure):")
print("=" * 60)
print(lehigh_raw.head())

print("\n" + "=" * 60)
print("LEHIGH COLUMNS:")
print("=" * 60)
print(list(lehigh_raw.columns))

## 2. Standardize Peer University Columns

Peer universities have Scopus data with standardized column names

In [ ]:
# Peer universities have clean Scopus data - no parsing needed!
# Just use the raw data directly
lehigh = lehigh_raw.copy()
marquette = marquette_raw.copy()
villanova = villanova_raw.copy()

print("✅ Peer university data loaded successfully!")
print(f"\nLehigh: {lehigh.shape[0]:,} publications, {lehigh.shape[1]} columns")
print(f"Marquette: {marquette.shape[0]:,} publications, {marquette.shape[1]} columns")
print(f"Villanova: {villanova.shape[0]:,} publications, {villanova.shape[1]} columns")

print(f"\nKey columns available:")
key_cols = ['Title', 'Authors', 'Year', 'Citations', 'Scopus Source title', 
            'Abstract', 'DOI', 'CiteScore (publication year)', 'SJR (publication year)']
print([col for col in key_cols if col in lehigh.columns])

In [ ]:
## 3. Compare Schemas

Check what columns AUB has vs peer universities

In [ ]:
print("=" * 60)
print("AUB COLUMNS:")
print("=" * 60)
print(list(aub.columns))
print(f"\nAUB shape: {aub.shape}")
display(aub.head(2))

print("\n" + "=" * 60)
print("PEER UNIVERSITIES COLUMNS (Scopus format):")
print("=" * 60)
print(f"\nTotal columns: {len(lehigh.columns)}")
print(f"\nFirst 15 columns:")
for i, col in enumerate(lehigh.columns[:15], 1):
    print(f"  {i:2d}. {col}")
print("  ...")
print(f"\nSample data:")
display(lehigh.head(2))

In [ ]:
## 4. Map Scopus Columns to AUB Schema

Create a standardized schema for comparison

In [ ]:
# Map Scopus columns to match AUB's schema
# Adjust this mapping based on what columns AUB actually has

scopus_to_standard = {
    'Title': 'title',
    'Authors': 'authors',
    'Year': 'year',
    'Citations': 'citations',
    'Scopus Source title': 'venue',
    'Abstract': 'abstract',
    'DOI': 'doi',
    'Publisher': 'publisher',
    'Source type': 'source_type',
    'Open Access': 'open_access',
    'CiteScore (publication year)': 'citescore',
    'SJR (publication year)': 'sjr',
    'Field-Weighted Citation Impact': 'field_weighted_citation_impact',
    'Number of Authors': 'num_authors',
    'Country/Region': 'country'
}

# Apply mapping to peer universities
lehigh_mapped = lehigh.rename(columns=scopus_to_standard)
marquette_mapped = marquette.rename(columns=scopus_to_standard)
villanova_mapped = villanova.rename(columns=scopus_to_standard)

print("✅ Column mapping applied!")
print(f"\nStandardized columns in peer data:")
print([col for col in scopus_to_standard.values() if col in lehigh_mapped.columns])

# Check which AUB columns match
aub_cols_set = set(aub.columns)
peer_cols_set = set(lehigh_mapped.columns)
common = aub_cols_set & peer_cols_set

print(f"\n📊 Columns common to both AUB and peers:")
print(sorted(common))

In [ ]:
## 5. Add Institution Tags & Merge

**ONLY RUN THIS AFTER** columns are properly mapped above!

In [ ]:
# Add institution tags
aub_tagged = aub.copy()
aub_tagged['institution'] = 'AUB'

lehigh_tagged = lehigh_mapped.copy()
lehigh_tagged['institution'] = 'Lehigh'

marquette_tagged = marquette_mapped.copy()
marquette_tagged['institution'] = 'Marquette'

villanova_tagged = villanova_mapped.copy()
villanova_tagged['institution'] = 'Villanova'

# Find common columns (essential ones for analysis)
all_cols = [aub_tagged.columns, lehigh_tagged.columns, 
            marquette_tagged.columns, villanova_tagged.columns]
common_cols = list(set.intersection(*[set(cols) for cols in all_cols]))

print(f"📊 Common columns across all 4 institutions:")
print(sorted(common_cols))
print(f"\nTotal: {len(common_cols)} common columns")

# If there are very few common columns, use a core set manually
if len(common_cols) < 5:
    print("\n⚠️  Few common columns found. Using core columns only:")
    core_cols = ['title', 'year', 'citations', 'institution']
    # Add optional columns if they exist in all datasets
    for col in ['authors', 'venue', 'abstract', 'doi']:
        if all(col in df.columns for df in [aub_tagged, lehigh_tagged, marquette_tagged, villanova_tagged]):
            core_cols.append(col)
    common_cols = core_cols
    print(common_cols)

# Merge all datasets
all_universities = pd.concat([
    aub_tagged[common_cols],
    lehigh_tagged[common_cols],
    marquette_tagged[common_cols],
    villanova_tagged[common_cols]
], ignore_index=True)

print(f"\n✅ MERGE COMPLETE!")
print(f"Total publications: {len(all_universities):,}")
print(f"Columns: {list(all_universities.columns)}")
print(f"\nBreakdown by institution:")
print(all_universities['institution'].value_counts())
print(f"\nSample merged data:")
display(all_universities.sample(3))

In [ ]:
## 6. Save Merged Dataset

In [ ]:
# Save merged dataset for future use
output_path = '../data/processed/all_universities.parquet'
all_universities.to_parquet(output_path, index=False)
print(f"✅ Saved to: {output_path}")

# Also save as CSV
csv_path = '../data/processed/all_universities.csv'
all_universities.to_csv(csv_path, index=False)
print(f"✅ Also saved CSV: {csv_path}")

print(f"\nDataset info:")
print(f"  - Shape: {all_universities.shape}")
print(f"  - Memory: {all_universities.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
## 7. Comparative Analysis

### 7.1 Citation Statistics

Calculate descriptive statistics for citations by institution

In [ ]:
# Calculate citation statistics by institution
citation_stats = all_universities.groupby('institution')['citations'].agg([
    'count',
    'mean',
    'median',
    'std',
    'min',
    'max',
    ('q25', lambda x: x.quantile(0.25)),
    ('q75', lambda x: x.quantile(0.75))
]).round(2)

print("=" * 60)
print("CITATION STATISTICS BY INSTITUTION")
print("=" * 60)
display(citation_stats)

# Visualization: Citation distribution comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box plot
ax1 = axes[0, 0]
all_universities.boxplot(column='citations', by='institution', ax=ax1)
ax1.set_title('Citation Distribution by Institution')
ax1.set_xlabel('Institution')
ax1.set_ylabel('Citations')
plt.sca(ax1)
plt.xticks(rotation=45)

# Violin plot
ax2 = axes[0, 1]
sns.violinplot(data=all_universities, x='institution', y='citations', ax=ax2)
ax2.set_title('Citation Density by Institution')
ax2.set_xlabel('Institution')
ax2.set_ylabel('Citations')
plt.sca(ax2)
plt.xticks(rotation=45)

# Bar chart - mean citations
ax3 = axes[1, 0]
citation_stats['mean'].plot(kind='bar', ax=ax3, color='skyblue')
ax3.set_title('Mean Citations by Institution')
ax3.set_xlabel('Institution')
ax3.set_ylabel('Mean Citations')
plt.sca(ax3)
plt.xticks(rotation=45)

# Publication counts
ax4 = axes[1, 1]
all_universities['institution'].value_counts().plot(kind='bar', ax=ax4, color='coral')
ax4.set_title('Number of Publications by Institution')
ax4.set_xlabel('Institution')
ax4.set_ylabel('Count')
plt.sca(ax4)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 7.2 Temporal Trends

In [ ]:
# Publications over time
yearly_pubs = all_universities.groupby(['institution', 'year']).size().reset_index(name='count')

plt.figure(figsize=(14, 6))
for inst in all_universities['institution'].unique():
    data = yearly_pubs[yearly_pubs['institution'] == inst]
    plt.plot(data['year'], data['count'], marker='o', label=inst)

plt.title('Publication Trends by Institution')
plt.xlabel('Year')
plt.ylabel('Number of Publications')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Average citations over time
yearly_citations = all_universities.groupby(['institution', 'year'])['citations'].mean().reset_index()

plt.figure(figsize=(14, 6))
for inst in all_universities['institution'].unique():
    data = yearly_citations[yearly_citations['institution'] == inst]
    plt.plot(data['year'], data['citations'], marker='o', label=inst)

plt.title('Average Citations per Publication by Institution Over Time')
plt.xlabel('Year')
plt.ylabel('Average Citations')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 7.3 High-Impact Publications

In [ ]:
# Define thresholds for high-impact papers
thresholds = [10, 25, 50, 100]

high_impact = pd.DataFrame()
for threshold in thresholds:
    counts = all_universities[all_universities['citations'] >= threshold].groupby('institution').size()
    high_impact[f'{threshold}+ citations'] = counts

high_impact = high_impact.fillna(0).astype(int)
high_impact

In [ ]:
# Percentage of high-impact papers
total_pubs = all_universities['institution'].value_counts()

high_impact_pct = pd.DataFrame()
for col in high_impact.columns:
    high_impact_pct[col] = (high_impact[col] / total_pubs * 100).round(2)

high_impact_pct

In [ ]:
# Visualization
high_impact_pct.plot(kind='bar', figsize=(12, 6))
plt.title('Percentage of High-Impact Publications by Institution')
plt.xlabel('Institution')
plt.ylabel('Percentage (%)')
plt.legend(title='Citation Threshold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Key Findings Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS: AUB vs Peer Universities")
print("=" * 60)
print(f"\nTotal Publications:")
print(all_universities['institution'].value_counts())
print(f"\nCitation Statistics:")
print(citation_stats[['mean', 'median', 'max']])
print(f"\nHigh-Impact Papers (100+ citations):")
print(high_impact['100+ citations'])
print(f"\nPercentage High-Impact:")
print(high_impact_pct['100+ citations'])

## 9. Next Steps

1. **Field/discipline analysis** - Compare citation patterns across research areas
2. **Author-level metrics** - h-index, productivity comparisons
3. **Venue quality** - Journal impact factors, conference rankings
4. **Collaboration patterns** - Co-authorship networks
5. **Geographic analysis** - International collaboration patterns